In [1]:
import json
import os
from datetime import datetime
import pytse_client as tse

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
def get_stock_info(symbol: str) -> dict:
    """
    اطلاعات لحظه‌ای یک نماد بورسی ایران را از TSETMC بازمی‌گرداند.

    Args:
        symbol (str): نماد سهم به فارسی یا انگلیسی، دقیقاً همانطور که در
            TSETMC ثبت شده است. مثال: "فولاد"

    Returns:
        dict: دیکشنری با کلیدهای زیر:
            - symbol (str): نماد سهم
            - name (str): نام کامل شرکت
            - last_price (float): آخرین قیمت معامله‌شده (تومان)
            - price_change_percent (float): درصد تغییر نسبت به روز قبل
            - volume (int): حجم معاملات
            - market_cap (float | None): ارزش بازار در صورت موجود بودن
            - last_update (str): زمان آخرین به‌روزرسانی به فرمت ISO 8601

    Raises:
        ValueError: اگر نماد وارد‌شده در TSETMC یافت نشود.
    """
    try:
        ticker = tse.Ticker(symbol)

        last_price = ticker.last_price
        yesterday_price = ticker.yesterday_price

        if yesterday_price and yesterday_price > 0:
            price_change_percent = round(
                ((last_price - yesterday_price) / yesterday_price) * 100, 2
            )
        else:
            price_change_percent = 0.0

        return {
            "symbol": symbol,
            "name": ticker.title,
            "last_price": last_price,
            "price_change_percent": price_change_percent,
            "volume": int(ticker.volume),
            "cap_market": None,
            "last_update": datetime.utcnow().isoformat()
        }
        
    except Exception as e:
        raise ValueError(
            f"{symbol} not found: {e}"
        )

# OpenAI

In [5]:
from openai import OpenAI

with open("openai_schema.json", "r", encoding="utf-8") as f:
    openai_tool_schema = json.load(f)

tool_mapping = {
    "get_stock_info": get_stock_info
}

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_API_URL")
)

messages = [
    {
        "role": "system",
        "content": """
        تو یک تحلیلگر مالی هوشمند هستی.
        
        هر زمان کاربر درباره قیمت، وضعیت یا تحلیل یک نماد بورسی ایران
        سؤال کرد، باید ابتدا از ابزار get_stock_info برای دریافت
        اطلاعات لحظه‌ای آن نماد استفاده کنی.
        
        هرگز قیمت، حجم معاملات یا درصد تغییر قیمت را حدس نزن.
        اطلاعات مورد نیاز را از نتیجه ابزار دریافت کن.
        
        پس از دریافت اطلاعات ابزار، پاسخ را بر اساس داده‌های واقعی
        ارائه کن. اگر اطلاعات کافی برای پاسخ وجود ندارد، این موضوع
        را شفاف اعلام کن.
        
        پاسخ‌ها را به زبان فارسی و به صورت واضح و مختصر ارائه کن.
        """
    },
    {
        "role": "user",
        "content": "وضعیت لحظه‌ای و آخرین قیمت سهم فولاد چطوره؟"
    }
]

response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    tools=[openai_tool_schema],
    tool_choice="auto"
)

response_message = response.choices[0].message
tool_calls = response_message.tool_calls

if tool_calls:
    messages.append(response_message)

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        function_to_call = tool_mapping[function_name]
        function_args = json.loads(tool_call.function.arguments)

        try:
            function_response = function_to_call(
                symbol=function_args.get("symbol")
            )
        except ValueError as e:
            function_response = {"error": str(e)}

        messages.append(
            {
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": json.dumps(function_response, ensure_ascii=False)
            }
        )

    second_response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
    )

    print(second_response.choices[0].message.content)

آخرین قیمت نماد "فولاد مبارکه اصفهان" (فولاد) برابر با ۴,۲۲۰ ریال است که نسبت به روز قبل ۲.۹۳ درصد افزایش داشته است. حجم معاملات این نماد نیز ۱,۰۰۶,۰۸۴,۵۰۹ سهم بوده است.


# Anthropic

In [10]:
import anthropic

with open("anthropic_schema.json", "r", encoding="utf-8") as f:
    anthropic_tool_schema = json.load(f)

client = anthropic.Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    # base_url=os.getenv("ANTHROPIC_API_URL"),
)

system_prompt = """
تو یک تحلیلگر مالی هوشمند هستی.

هر زمان کاربر درباره قیمت، وضعیت یا تحلیل یک نماد بورسی ایران
سؤال کرد، باید ابتدا از ابزار get_stock_info برای دریافت
اطلاعات لحظه‌ای آن نماد استفاده کنی.

هرگز قیمت، حجم معاملات یا درصد تغییر قیمت را حدس نزن.
اطلاعات مورد نیاز را از نتیجه ابزار دریافت کن.

پس از دریافت اطلاعات ابزار، پاسخ را بر اساس داده‌های واقعی
ارائه کن. اگر اطلاعات کافی برای پاسخ وجود ندارد، این موضوع
را شفاف اعلام کن.

پاسخ‌ها را به زبان فارسی و به صورت واضح و مختصر ارائه کن.
"""

messages = [
    {
        "role": "user",
        "content": "وضعیت لحظه‌ای و آخرین قیمت سهم فولاد چطوره؟"
    }
]

response = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=1024,
    system=system_prompt,
    messages=messages,
    tools=[anthropic_tool_schema],
    tool_choice={"type": "auto"}
)

response_content = response.content

tool_calls = [
    block for block in response_content
    if block.type == "tool_use"
]

if tool_calls:
    messages.append(
        {
            "role": "assistant",
            "content": response_content
        }
    )

    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.name
        function_to_call = tool_mapping[function_name]
        function_args = tool_call.input

        try:
            function_response = function_to_call(
                symbol=function_args.get("symbol")
            )
        except ValueError as e:
            function_response = {"error": str(e)}

        tool_results.append(
            {
                "type": "tool_result",
                "tool_use_id": tool_call.id,
                "content": json.dumps(
                    function_response,
                    ensure_ascii=False
                )
            }
        )

    messages.append(
        {
            "role": "user",
            "content": tool_results
        }
    )

    second_response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        system=system_prompt,
        messages=messages
    )

    final_text = "\n".join(
        block.text
        for block in second_response.content
        if block.type == "text"
    )

    print(final_text)

TypeError: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"